<a href="https://colab.research.google.com/github/nandayot/RNA/blob/main/RNA_Trabalho_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Entrega:

    Repositório GitHub com código-fonte e instruções de execução.
    Vídeo (5–10 minutos) demonstrando o funcionamento do sistema e explicando as etapas do projeto.

Objetivo

Desenvolver um mini sistema de Retrieval-Augmented Generation (RAG) que una recuperação de informação e geração de texto baseada em modelos de linguagem.
O trabalho deve incluir uma avaliação de factualidade das respostas utilizando o Fact Score.

Descrição Geral

O aluno (individualmente ou em dupla) deverá implementar um pipeline simples de RAG composto por:

    1. Base de conhecimento: regulamento da pós-graduação acadêmica da FACOM.
    2. Indexação vetorial: criação de embeddings e armazenamento no ChromaDB.
    3. Recuperação de contexto: busca semântica (top-k) de trechos relevantes para uma pergunta.
    4. Geração de resposta: uso de um modelo generativo (ex.: GPT-3.5, Llama-3, Flan-T5, Mistral).
    5. Avaliação de factualidade: cálculo do Fact Score para medir a consistência das respostas com os documentos recuperados.
    6. Apresentação: vídeo explicando o funcionamento, a arquitetura e os resultados obtidos.

Etapas Recomendadas
1. Preparação e Indexação

    Escolher e preparar os textos.\
    Gerar embeddings (ex.: sentence-transformers).\
    Indexar no ChromaDB.

2. Recuperação e Geração

    Implementar busca vetorial (função de consulta).\
    Recuperar os contextos mais relevantes.\
    Montar o prompt com contexto + pergunta e gerar a resposta via LLM.

3. Avaliação e Apresentação

    Calcular o Fact Score comparando a resposta com os contextos recuperados.\
    Registrar os resultados (tabela, gráfico ou média).
    Produzir vídeo explicando o pipeline, as decisões técnicas e a análise dos resultados.

Neste trabalho é permitido o uso de LLMs para ajuda na codificação do trabalho.

In [ ]:
# https://blog.langchain.com/langchain-chroma/
# https://docs.langchain.com/oss/python/langchain/overview
# https://github.com/grumpyp/chroma-langchain-tutorial
# https://github.com/hwchase17/chroma-langchain/blob/master/qa.ipynb
# https://docs.langchain.com/oss/javascript/integrations/vectorstores/chroma
# https://github.com/tecepeipe/ollama-colab-runner/blob/main/ollama_colab_runner.ipynb
# https://docs.langchain.com/oss/python/integrations/text_embedding/ollama
# https://github.com/ollama/ollama
# https://docs.ragas.io/en/stable/references/evaluate/#ragas.evaluation.evaluate

In [ ]:
import os
import re
import math
import subprocess

In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-core
!pip install chromadb
!pip install ollama
!pip install langchain-ollama
!pip install langchain-text-splitters
!pip install langchain-classic
!pip install ragas datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
'''
!pip install langchain-classic
!pip install --upgrade langchain
!pip install --upgrade langchain-core
!pip install --upgrade langchain-community
!pip install --upgrade langchain-ollama
!pip install "langchain-chroma>=0.1.2"
'''

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 22.5 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.27
    Uninstalling langchain-0.3.27:
      Successfully uninstalled langchain-0.3.27


In [ ]:
!curl https://ollama.ai/install.sh | sh

!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get update && sudo apt-get install -y cuda-drivers

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  58994      0 --:--:-- --:--:-- --:--:-- 59026
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com

In [ ]:
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

In [ ]:
proccess = subprocess.Popen(['ollama', 'serve'])

In [ ]:
!ollama pull "mxbai-embed-large"

In [ ]:
!ollama pull "gemma3:27b"

Error: ollama server not responding - could not connect to ollama server, run 'ollama serve' to start it



In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_classic.chains import RetrievalQA
from langchain_community.document_loaders import TextLoader
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import OllamaLLM
from langchain_classic.schema import Document
from typing import List
from ragas import evaluate
from ragas.metrics import faithfulness
from ragas.run_config import RunConfig
from datasets import Dataset
from IPython.display import display, Markdown, display_markdown
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_classic.globals import set_debug

# Preparação e Indexação

In [ ]:
doc_path = "/content/drive/MyDrive/Regulamento-Pós-Graduação-Acadêmico-2023.txt"

In [ ]:
loader = TextLoader(doc_path)
documents = loader.load()

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=0)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

In [ ]:
store = InMemoryStore()

In [ ]:
llm = OllamaLLM(model="gemma3:27b")
llm_ragas = OllamaLLM(model="gemma3:27b", format="json", system="You are a strict JSON generator. Output ONLY valid JSON. Do not write explanations.")

In [ ]:
embeddings = OllamaEmbeddings(model='mxbai-embed-large')

In [ ]:
vectorstore = Chroma(
    collection_name="split_parents_children",
    embedding_function=embeddings
)

/tmp/ipython-input-3669243571.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


# Recuperação e Geração

<small>Um tipo de recuperador de documentos que divide os documentos de entrada em partes menores, armazenando e preservando separadamente os documentos originais. As partes menores são incorporadas e, na recuperação, os documentos originais ("pais") são recuperados.</small>

<small>Isso estabelece um equilíbrio entre uma recuperação mais precisa com documentos pequenos e documentos maiores, que contêm mais contexto.</small>

In [ ]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 2}
)

In [ ]:
retriever.add_documents(documents)

In [ ]:
def callRAG(query):
    retrieved_docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    prompt_template = f"""
    Use o contexto abaixo para responder a pergunta.
    Se a resposta não estiver no contexto, apenas diga que não sabe.

    Contexto:
    {context}

    Pergunta:
    {query}

    Resposta:
    """
    response = llm.invoke(prompt_template)
    return response, retrieved_docs

In [ ]:
queries = [
    'Qual resolução foi revogada pela Resolução Nº 704-COPP/UFMS?',
    'Quando a Resolução Nº 704-COPP/UFMS entrou em vigor?',
    'Qual é o prazo máximo (regular) para a conclusão do curso de Mestrado?',
    'Qual é o prazo máximo (regular) para a conclusão do curso de Doutorado?',
    'Quantos orientandos de mestrado e doutorado um único professor pode ter simultaneamente?',
    'Quais são os dois requisitos para um docente poder orientar um estudante de doutorado?',
    'Qual é a pontuação mínima total, e a pontuação mínima dos itens 1 a 5, para que um professor seja credenciado como docente permanente?',
    'Docentes classificados como Sênior ou Júnior precisam cumprir os critérios de pontuação para permanência?',
    'O que acontece com um docente permanente que não cumpre os critérios de permanência, mas ainda tem alunos para orientar?',
    'Qual era a pontuação mínima exigida para a permanência de docentes permanentes no ano de 2023?',
    'Quantos créditos no total um estudante de Mestrado precisa obter para a diplomação?',
    'Quantos créditos no total um estudante de Doutorado precisa obter para a diplomação?',
    'Um artigo científico publicado antes do ingresso no curso pode contar como créditos? Se sim, quais as condições?',
    'Créditos cursados no Mestrado podem ser aproveitados (convalidados) no Doutorado?',
    'O Estágio de Docência é obrigatório para todos os alunos de mestrado e doutorado?',
    'Qual é o prazo limite para um estudante de Mestrado comprovar sua proficiência em língua inglesa?',
    'Qual é o prazo limite para um estudante de Doutorado comprovar sua proficiência em língua inglesa?',
    'Um "estudante ouvinte" tem registro de notas e faltas?',
    'O que acontece se um estudante reprovar em qualquer disciplina no seu primeiro semestre letivo?',
    'Qual é o coeficiente de rendimento (CR) mínimo que um estudante deve ter ao final do seu primeiro semestre para não ser desligado?',
    'A partir do segundo semestre, qual o coeficiente de rendimento (CR) mínimo exigido?',
    'Quais são os critérios necessários para um estudante de Mestrado solicitar a defesa da dissertação?',
    'Quais são os critérios necessários para um estudante de Doutorado solicitar a defesa da tese?',
    'Como deve ser composta a banca examinadora da defesa de Doutorado? (Quantos membros, quantos externos, etc.)',
    'O cônjuge de um estudante pode participar da sua banca de qualificação ou defesa?',
    'Considerando o Item 6 da tabela, se um professor orientar 5 alunos de Iniciação Científica (PIBIC) em um único ano, quantos pontos ele receberá efetivamente, levando em conta o limite anual estabelecido?',
    'Qual atividade gera maior pontuação individual por item: a aceitação de uma patente (Item 8) ou a premiação de um artigo no Congresso da Sociedade Brasileira de Computação - CSBC (Item 10)?',
    'Qual é a diferença de pontuação entre publicar um artigo em um periódico classificado como A1 e um classificado como A4 no Qualis da Capes?',
    'Existe algum limite anual de pontuação para a publicação de artigos técnico-científicos em periódicos ou reuniões científicas classificados nos estratos A1, A2, A3 ou A4 (Itens 2 a 5)?',
    'Para fins de pontuação no Item 15, qualquer coordenação de evento científico é válida ou existe algum critério específico relacionado à classificação do evento no Qualis da Capes?'
]

<small>Perguntas foram geradas pelo Gemini</small>

# Avaliação e Apresentação

In [ ]:
def faithfulnessEvaluation(query, response, retrieved_docs):
    print("\n--- INICIANDO AVALIAÇÃO DE FATOS ---")
    contexts_list = [doc.page_content for doc in retrieved_docs]
    data_samples = {
        'question': [query],
        'answer': [response],
        'contexts': [contexts_list],
    }
    dataset = Dataset.from_dict(data_samples)

    minha_config = RunConfig(
        timeout=300,
        max_retries=3
    )

    score = evaluate(
        dataset,
        metrics=[faithfulness],
        llm=llm_ragas,
        embeddings=embeddings,
        raise_exceptions=False,
        run_config=minha_config
    )

    df_resultado = score.to_pandas()
    print("--- RESULTADO DA AVALIAÇÃO (FACTSCORE) ---")
    if 'faithfulness' in df_resultado.columns:
        pontuacao = df_resultado['faithfulness'].iloc[0]
        pontuacao_text = f"Pontuação de Fidelidade (FactScore): {pontuacao:.4f}"
        display(Markdown(pontuacao_text))
        return pontuacao
    else:
        print(f"A métrica 'faithfulness' não foi encontrada. Colunas: {df_resultado.columns}")

In [ ]:
def format_response(query, response):
    return f"""\n
### PERGUNTA
{query}
### RESPOSTA
{response}
"""

## Chamada Principal

In [ ]:
notas_factscore = []
print(f"--- INICIANDO AVALIAÇÃO DE {len(queries)} PERGUNTAS ---\n")
for i, query in enumerate(queries):
    display(Markdown(f"### Pergunta {i+1}: {query}"))
    response, retrieved_docs = callRAG(query)
    display(Markdown(f"**Resposta:**\n{response}"))
    nota_atual = faithfulnessEvaluation(query, response, retrieved_docs)
    notas_factscore.append(nota_atual)
    print("-" * 40)

--- INICIANDO AVALIAÇÃO DE 30 PERGUNTAS ---



### Pergunta 1: Qual resolução foi revogada pela Resolução Nº 704-COPP/UFMS?

**Resposta:**
Resolução nº 134, de 15 de julho de 2019.


--- INICIANDO AVALIAÇÃO DE FATOS ---


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

...


--- INICIANDO AVALIAÇÃO DE FATOS ---


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

--- RESULTADO DA AVALIAÇÃO (FACTSCORE) ---


Pontuação de Fidelidade (FactScore): 1.0000

----------------------------------------


## Relatório

In [ ]:
print("\n" + "="*30)
print("RELATÓRIO FINAL DE FACTSCORE")
print("="*30)

if len(notas_factscore) > 0:
    notas_validas = [float(nota) for nota in notas_factscore if not math.isnan(nota)]
    n_ignorados = len(notas_factscore) - len(notas_validas)

    media_final = sum(notas_validas) / len(notas_validas)

    if n_ignorados > 0:
      print(f"Atenção: {n_ignorados} avaliações resultaram em erro (NaN) e foram ignoradas.")
    notas_formatadas = [round(n, 2) for n in notas_validas]
    print(f"Notas individuais: {notas_formatadas}")
    print(f"Total de perguntas avaliadas: {len(notas_validas)}")
    print(f"\nMÉDIA GERAL (FactScore): {media_final:.4f}")

    if media_final > 0.9:
        print("Classificação: Excelente (Altamente Factual)")
    elif media_final > 0.7:
        print("Classificação: Bom (Geralmente Factual)")
    else:
        print("Classificação: Cuidado (Possíveis Alucinações)")
else:
    print("Nenhuma nota foi registrada.")


RELATÓRIO FINAL DE FACTSCORE
Atenção: 3 avaliações resultaram em erro (NaN) e foram ignoradas.
Notas individuais: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0]
Total de perguntas avaliadas: 27

MÉDIA GERAL (FactScore): 0.7037
Classificação: Bom (Geralmente Factual)


## Gemini + Debug

<small>Foi feito um teste utilizando o <i><b>Gemini Pro + Gems</b></i> para criar dois agentes específicos. Um para a tarefa de criar sentenças menores de acordo com a resposta que o modelo <i><b>Gemma</b></i> gerou e outro para avaliar as sentenças e calcular <i><b>faithfulness</b></i> de acordo com o contexto fornecido.</small>

<small>Isso foi feito porque tiveram algumas perguntas que o avaliador não conseguiu calcular devido a timeout. As chamadas foram feitas novamente e por isso a resposta gerada pode ter sido diferente dos exemplos acima. Com o debug ligado é possível extrair os prompts que foi utilizado para criar os avaliadores e isso foi embutido nos Gems do Gemini. As respostas dos avaliadores foi extraída para as 30 perguntas e a métrica foi calculada da mesma forma que acima. Segue relatório:</small>

#### RELATÓRIO FINAL DE FACTSCORE (com Gemini)

Notas individuais: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.3, 1.0, 1.0, 0.0, 0.0, 0.6, 0.83, 1.0, 1.0, 0.6, 1.0, 1.0, 0.6, 1.0]\
Total de perguntas avaliadas: 30

MÉDIA GERAL (FactScore): 0.8477\
Classificação: Bom (Geralmente Factual)

In [ ]:
notas_factscore = []
response_arr = []
retrieved_docs_arr = []
print(f"--- INICIANDO AVALIAÇÃO DE {len(queries)} PERGUNTAS ---\n")
for i, query in enumerate(queries):
    display(Markdown(f"### Pergunta {i+1}: {query}"))
    response, retrieved_docs = callRAG(query)
    response_arr.append(response)
    retrieved_docs_arr.append(retrieved_docs)
    display(Markdown(f"**Resposta:**\n{response}"))
    set_debug(True)
    nota_atual = faithfulnessEvaluation(query, response, retrieved_docs)
    notas_factscore.append(nota_atual)
    set_debug(False)
    print("-" * 40)

--- INICIANDO AVALIAÇÃO DE 30 PERGUNTAS ---



### Pergunta 1: Qual resolução foi revogada pela Resolução Nº 704-COPP/UFMS?

**Resposta:**
Resolução nº 134, de 15 de julho de 2019.


--- INICIANDO AVALIAÇÃO DE FATOS ---
[chain/start] [chain:ragas evaluation] Entering Chain run with input:
{}
[chain/start] [chain:ragas evaluation > chain:row 0] Entering Chain run with input:
{
  "user_input": "Qual resolução foi revogada pela Resolução Nº 704-COPP/UFMS?",
  "retrieved_contexts": [
    "RESOLUÇÃO Nº 704-COPP/UFMS, DE 7 DE JULHO DE 2023.\nAprova o Regulamento dos Cursos de Mestrado e Doutorado em Ciência da Computação da Faculdade de Computação.\n\nA PRESIDENTE DO CONSELHO DE PESQUISA E PÓS-GRADUAÇÃO da Fundação Universidade Federal de Mato Grosso do Sul, no uso da atribuição que lhe confere o art. 5º, caput, inciso IV, do Regimento Geral da UFMS, e considerando o contido no Processo nº 23104.018154/2023-90, resolve, ad referendum:\n\nArt. 1º Aprovar o Regulamento dos Cursos de Mestrado e Doutorado em Ciência da Computação, da Faculdade de Computação, na forma do Anexo a esta Resolução.\nArt. 2º Fica revogada a Resolução nº 134, de 15 de julho de 2019.\nArt. 3º Esta 

/usr/local/lib/python3.12/dist-packages/pydantic/v1/main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


...


--- INICIANDO AVALIAÇÃO DE FATOS ---
[chain/start] [chain:ragas evaluation] Entering Chain run with input:
{}
[chain/start] [chain:ragas evaluation > chain:row 0] Entering Chain run with input:
{
  "user_input": "Para fins de pontuação no Item 15, qualquer coordenação de evento científico é válida ou existe algum critério específico relacionado à classificação do evento no Qualis da Capes?",
  "retrieved_contexts": [
    "13 | Editor Chefe ou Associado de revista científica com classificação no estrato restrito do Qualis da Capes. | 20 Pts/Revista | 40 Pts\n14 | Coordenação de comitê de programa de evento científico com classificação no estrato restrito do Qualis da Capes. | 15 Pts/Evento | 30 Pts\f15 | Coordenação geral de evento científico com classificação no estrato restrito do Qualis da Capes. | 15 Pts/Evento | 30 Pts\n16 | Membro de comitê científico de conferência com classificação no estrato restrito do Qualis da Capes. | 10 Pts/Evento | 20 Pts\n17 | Premiação de artigo public

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

[chain/start] [chain:ragas evaluation > chain:row 0 > chain:faithfulness] Entering Chain run with input:
{
  "user_input": "Para fins de pontuação no Item 15, qualquer coordenação de evento científico é válida ou existe algum critério específico relacionado à classificação do evento no Qualis da Capes?",
  "retrieved_contexts": [
    "13 | Editor Chefe ou Associado de revista científica com classificação no estrato restrito do Qualis da Capes. | 20 Pts/Revista | 40 Pts\n14 | Coordenação de comitê de programa de evento científico com classificação no estrato restrito do Qualis da Capes. | 15 Pts/Evento | 30 Pts\f15 | Coordenação geral de evento científico com classificação no estrato restrito do Qualis da Capes. | 15 Pts/Evento | 30 Pts\n16 | Membro de comitê científico de conferência com classificação no estrato restrito do Qualis da Capes. | 10 Pts/Evento | 20 Pts\n17 | Premiação de artigo publicado em conferência com classificação no estrato restrito do Qualis da Capes. | 10 Pts/arti

Pontuação de Fidelidade (FactScore): 1.0000

----------------------------------------
